# RK1828 Whisper-base 语音识别 Demo


OpenAI [whisper-base](https://github.com/openai/whisper)，中英文语音识别。
拆成三个模型：**编码器** + **首 token 解码器（decode0）** + **自回归解码器（decode1）**，
实测中文示例 RTF 0.073（比实时快约 14 倍）。

| 步骤 | 内容 |
|---|---|
| 1 | 下载模型和示例音频（可选 HF / HF-Mirror 源） |
| 2 | 解压预编译 demo |
| 3 | 选择语种和音频 |
| 4 | 识别 |

## 0. 环境检查

先确认 RKNN3 运行库（`librknn3_api.so`）和 RK1828 设备都正常。
如失败，请先安装驱动/运行时（`rknn3-rk182x` DEB 包），并用 `sudo rknn-smi info` 确认能看到 RK1828。

In [ ]:
from rknn3 import get_lib, find_devices

_ = get_lib()  # 加载 /usr/lib/librknn3_api.so；失败说明 Runtime 未安装

devices = find_devices()
if not devices:
    raise RuntimeError("未发现 RK1828 设备！请检查: sudo rknn-smi info")
for dev_id, dev_type in devices:
    print(f"发现设备: id={dev_id}  接口={dev_type}")

## 1. 下载模型

本 demo 的模型文件托管在 Hugging Face（`qhddz` 账号）。
**先用下拉框选择下载源**，再运行下一个单元格开始下载：

- **HF-Mirror 国内镜像**：`hf-mirror.com`，国内直连，推荐
- **Hugging Face 官方**：`huggingface.co`，海外或已有代理时用

> 已存在（且非空）的文件会**自动跳过**，重复运行不会重新下载。

In [ ]:
from nb_download import source_picker

# (HF仓库名, 仓库内路径, 本地路径)
FILES = [
    ("whisper-base-rknn3-rk1828", "demo_x86_64.tar.gz", "demos/whisper_base/demo_x86_64.tar.gz"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "demos/whisper_base/model/long_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/mel_128_filters.txt", "demos/whisper_base/model/mel_128_filters.txt"),
    ("whisper-base-rknn3-rk1828", "model/mel_80_filters.txt", "demos/whisper_base/model/mel_80_filters.txt"),
    ("whisper-base-rknn3-rk1828", "model/position_embed.bin", "demos/whisper_base/model/position_embed.bin"),
    ("whisper-base-rknn3-rk1828", "model/prompt_embed.bin", "demos/whisper_base/model/prompt_embed.bin"),
    ("whisper-base-rknn3-rk1828", "model/prompt_embed_en.bin", "demos/whisper_base/model/prompt_embed_en.bin"),
    ("whisper-base-rknn3-rk1828", "model/prompt_embed_zh.bin", "demos/whisper_base/model/prompt_embed_zh.bin"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "demos/whisper_base/model/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "demos/whisper_base/model/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/token_embed.bin", "demos/whisper_base/model/token_embed.bin"),
    ("whisper-base-rknn3-rk1828", "model/vocab_en.txt", "demos/whisper_base/model/vocab_en.txt"),
    ("whisper-base-rknn3-rk1828", "model/vocab_zh.txt", "demos/whisper_base/model/vocab_zh.txt"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode0_base.rknn", "demos/whisper_base/model/whisper_decode0_base.rknn"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode0_base.weight", "demos/whisper_base/model/whisper_decode0_base.weight"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode1_base.rknn", "demos/whisper_base/model/whisper_decode1_base.rknn"),
    ("whisper-base-rknn3-rk1828", "model/whisper_decode1_base.weight", "demos/whisper_base/model/whisper_decode1_base.weight"),
    ("whisper-base-rknn3-rk1828", "model/whisper_encoder_base.rknn", "demos/whisper_base/model/whisper_encoder_base.rknn"),
    ("whisper-base-rknn3-rk1828", "model/whisper_encoder_base.weight", "demos/whisper_base/model/whisper_encoder_base.weight"),
    ("whisper-base-rknn3-rk1828", "model/test_zh.wav", "datasets/asr/test_zh.wav"),
    ("whisper-base-rknn3-rk1828", "model/test_en.wav", "datasets/asr/test_en.wav"),
    ("whisper-base-rknn3-rk1828", "model/long_en.wav", "datasets/asr/long_en.wav"),
    ("sensevoice-small-rknn3-rk1828", "model/zh.wav", "datasets/asr/sensevoice_zh.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example.wav", "datasets/asr/asr_example.wav"),
    ("paraformer-rknn3-rk1828", "model/asr_example_hotword.wav", "datasets/asr/asr_example_hotword.wav"),
    ("qwen3-forced-aligner-rknn3-rk1828", "audio.wav", "datasets/asr/aligner_demo.wav"),
]

w_source = source_picker()

In [ ]:
from nb_download import ensure_files

ensure_files(FILES, w_source.value)

## 2. 解压预编译 demo

`demo_x86_64.tar.gz` 里是官方流程编译好的 x86_64 C++ demo 和运行时库（`lib/`）。
HF 不允许直接托管可执行文件，所以打包成 tar.gz，解压一次即可（重复运行自动跳过）。

In [ ]:
from nb_download import extract_tarball

extract_tarball("demos/whisper_base/demo_x86_64.tar.gz", "demos/whisper_base")

In [ ]:
from nb_demo import run_demo, snapshot, new_files, show_images, show_new_images, show_audio

## 3. 选择语种和音频

In [ ]:
import glob
import ipywidgets as widgets
from IPython.display import display

w_lang = widgets.Dropdown(options=[("中文", "zh"), ("English", "en")],
                          value="zh", description="语种:")
w_wav = widgets.Dropdown(options=sorted(glob.glob("datasets/asr/*.wav")),
                         description="音频:", layout=widgets.Layout(width="60%"))
if any(o.endswith('/test_zh.wav') for o in w_wav.options):
    w_wav.value = [o for o in w_wav.options if o.endswith('/test_zh.wav')][0]
display(w_lang, w_wav)
print("选好后继续往下运行 ▼")

先听一下这段音频：

In [ ]:
from nb_demo import show_audio

show_audio(w_wav.value)

## 4. 识别

In [ ]:
import os

DEMO = "demos/whisper_base"
wav = os.path.abspath(w_wav.value)
run_demo(["./rknn_whisper_raw_demo",
          "model/whisper_encoder_base.rknn", "model/whisper_encoder_base.weight",
          "model/whisper_decode0_base.rknn", "model/whisper_decode0_base.weight",
          "model/whisper_decode1_base.rknn", "model/whisper_decode1_base.weight",
          w_lang.value, wav, "model/token_embed.bin",
          "0xff", "0xff", "0xff", "--model-type", "base"], DEMO)

## 小结

- Whisper 的自回归解码被拆成"首 token"和"后续 token"两个模型，是 NPU 部署 Transformer
  decoder 的常见优化（首 token 不需要 KV cache）。
- 长音频（如 `long_en.wav`）会被自动切分处理。